# Практическое занятие 4  
## Основы SQL: работа с несколькими таблицами, JOIN

### Тема занятия
**"Основы работы с SQL. Связи таблиц, JOIN-запросы и итоговый анализ данных"**

## Цель занятия

Научиться работать с несколькими связанными таблицами и выполнять аналитические SQL-запросы с использованием `JOIN`.

После занятия слушатели должны понимать:

1. зачем данные разделяют на несколько таблиц;
2. что такое связь между таблицами;
3. что такое первичный и внешний ключ;
4. как работает `INNER JOIN`;
5. как работает `LEFT JOIN`;
6. как находить записи без связанных данных;
7. как рассчитывать аналитические показатели по нескольким таблицам;
8. как формулировать выводы по результатам SQL-запросов.

# План пары

| Этап | Время | Содержание |
|---|---:|---|
| 1 | 10 минут | Повторение SQL по одной таблице |
| 2 | 10 минут | Создание таблиц `users` и `orders` |
| 3 | 15 минут | Расчёт суммы заказа |
| 4 | 20 минут | `INNER JOIN` и анализ заказов пользователей |
| 5 | 15 минут | `LEFT JOIN` и поиск пользователей без заказов |
| 6 | 15 минут | Итоговый мини-кейс |
| 7 | 5 минут | Итоги и контрольные вопросы |

---

## Связь с предыдущими парами

На первой паре мы научились загружать данные из CSV и JSON.

На второй паре мы извлекали данные из HTML и приводили их к табличному виду.

На третьей паре мы изучили SQL на одной таблице `users`: `SELECT`, `WHERE`, `ORDER BY`, агрегатные функции и `GROUP BY`.

На четвёртой паре мы переходим к более реалистичной ситуации: данные хранятся не в одной таблице, а в нескольких связанных таблицах.

# 1. Повторение: что мы уже знаем

На прошлой паре мы работали с таблицей `users`.

Мы уже умеем:

```sql
SELECT *
FROM users;
```

```sql
SELECT *
FROM users
WHERE country = 'RUS';
```

```sql
SELECT country, AVG(balance)
FROM users
GROUP BY country;
```

Теперь добавим вторую таблицу — `orders`.

Она будет хранить заказы пользователей.

# 2. Почему данные разделяют на несколько таблиц

В реальных базах данных редко хранят всё в одной большой таблице.

Например, у нас есть пользователи и их заказы.

Можно было бы сделать одну большую таблицу:

| user_name | email | age | product_name | price | quantity |
|---|---|---:|---|---:|---:|

Но это приведёт к проблемам:

1. данные пользователя будут повторяться в каждой строке заказа;
2. таблица станет громоздкой;
3. возраст, email или имя придётся обновлять сразу в нескольких строках;
4. появится риск ошибок и дублирования.

Поэтому данные обычно разделяют:

- `users` — данные о пользователях;
- `orders` — данные о заказах.

Связь между ними строится через ключи:

- `users.id` — идентификатор пользователя;
- `orders.user_id` — идентификатор пользователя, который сделал заказ.

# 3. Основные понятия

| Понятие | Простое объяснение |
|---|---|
| Первичный ключ | уникальный идентификатор строки в таблице |
| Внешний ключ | поле, которое ссылается на строку в другой таблице |
| Связь таблиц | логическое соединение данных из разных таблиц |
| JOIN | SQL-оператор для объединения таблиц |
| INNER JOIN | показывает только совпадающие записи |
| LEFT JOIN | показывает все записи из левой таблицы и совпадения из правой |
| NULL | отсутствие значения |

Сегодня мы будем использовать две таблицы:

1. `users`;
2. `orders`.

# 4. Подготовка рабочей среды

Как и на третьей паре, используем SQLite прямо в Google Colab.

Это позволяет выполнять SQL-запросы без установки MySQL, Docker или отдельного сервера.

In [ ]:
import sqlite3
import pandas as pd

In [ ]:
# Создаём базу данных в оперативной памяти.

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()

print('База данных SQLite создана успешно.')

База данных SQLite создана успешно.


In [ ]:
# Функция для выполнения SELECT-запросов и отображения результата в виде таблицы pandas.

def run_sql(query):
    return pd.read_sql_query(query, connection)

# 5. Создание таблицы users

Сначала создадим таблицу пользователей.

Она уже знакома нам по третьей паре.

In [ ]:
cursor.execute('''
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    age INTEGER NOT NULL,
    country TEXT NOT NULL,
    balance REAL NOT NULL
);
''')

connection.commit()

print('Таблица users создана успешно.')

Таблица users создана успешно.


In [ ]:
cursor.execute('''
INSERT INTO users (id, name, email, age, country, balance) VALUES
(1, 'Ivan', 'ivan@example.com', 25, 'RUS', 100000),
(2, 'Gleb', 'gleb@example.com', 26, 'RUS', 90000),
(3, 'Sergey', 'sergey@example.com', 28, 'RUS', 90000),
(4, 'Andrew', 'andrew@example.com', 24, 'RUS', 95000),
(5, 'Bob', 'bob@example.com', 25, 'USA', 100000),
(6, 'Tom', 'tom@example.com', 29, 'USA', 110000);
''')

connection.commit()

print('Данные в users добавлены успешно.')

Данные в users добавлены успешно.


Проверим таблицу `users`.

In [ ]:
run_sql('''
SELECT *
FROM users;
''')

,id,name,email,age,country,balance
0,1,Ivan,ivan@example.com,25,RUS,100000.0
1,2,Gleb,gleb@example.com,26,RUS,90000.0
2,3,Sergey,sergey@example.com,28,RUS,90000.0
3,4,Andrew,andrew@example.com,24,RUS,95000.0
4,5,Bob,bob@example.com,25,USA,100000.0
5,6,Tom,tom@example.com,29,USA,110000.0


# 6. Создание таблицы orders

Теперь создадим таблицу заказов.

В ней будут поля:

| Поле | Тип данных | Описание |
|---|---|---|
| id | INTEGER | уникальный идентификатор заказа |
| user_id | INTEGER | идентификатор пользователя |
| product_name | TEXT | название товара |
| price | REAL | цена товара |
| quantity | INTEGER | количество |
| order_date | TEXT | дата заказа |

Поле `user_id` показывает, какой пользователь сделал заказ.

In [ ]:
cursor.execute('''
CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    product_name TEXT NOT NULL,
    price REAL NOT NULL,
    quantity INTEGER NOT NULL,
    order_date TEXT NOT NULL
);
''')

connection.commit()

print('Таблица orders создана успешно.')

Таблица orders создана успешно.


In [ ]:
cursor.execute('''
INSERT INTO orders (id, user_id, product_name, price, quantity, order_date) VALUES
(1, 1, 'Product A', 10, 2, '2023-02-22'),
(2, 2, 'Product B', 20, 1, '2023-02-20'),
(3, 2, 'Product C', 15, 3, '2023-02-23'),
(4, 1, 'Product D', 12, 2, '2023-02-25'),
(5, 2, 'Product E', 25, 1, '2023-02-26');
''')

connection.commit()

print('Данные в orders добавлены успешно.')

Данные в orders добавлены успешно.


Проверим таблицу `orders`.

In [ ]:
run_sql('''
SELECT *
FROM orders;
''')

,id,user_id,product_name,price,quantity,order_date
0,1,1,Product A,10.0,2,2023-02-22
1,2,2,Product B,20.0,1,2023-02-20
2,3,2,Product C,15.0,3,2023-02-23
3,4,1,Product D,12.0,2,2023-02-25
4,5,2,Product E,25.0,1,2023-02-26


# 7. Проверяем связь между таблицами

В таблице `users` есть поле `id`.

В таблице `orders` есть поле `user_id`.

Именно через эти поля таблицы связываются:

```text
users.id = orders.user_id
```

Например:

- если `orders.user_id = 1`, значит заказ сделал пользователь с `users.id = 1`;
- если `orders.user_id = 2`, значит заказ сделал пользователь с `users.id = 2`.

Проверим пользователей с `id = 1` и `id = 2`.

In [ ]:
run_sql('''
SELECT *
FROM users
WHERE id IN (1, 2);
''')

,id,name,email,age,country,balance
0,1,Ivan,ivan@example.com,25,RUS,100000.0
1,2,Gleb,gleb@example.com,26,RUS,90000.0


Именно эти пользователи есть в таблице заказов.

In [ ]:
run_sql('''
SELECT *
FROM orders
WHERE user_id IN (1, 2);
''')

,id,user_id,product_name,price,quantity,order_date
0,1,1,Product A,10.0,2,2023-02-22
1,2,2,Product B,20.0,1,2023-02-20
2,3,2,Product C,15.0,3,2023-02-23
3,4,1,Product D,12.0,2,2023-02-25
4,5,2,Product E,25.0,1,2023-02-26


# 8. Расчёт суммы заказа

В таблице `orders` есть:

- `price` — цена товара;
- `quantity` — количество.

Сумма заказа рассчитывается так:

```text
price * quantity
```

В SQL можно создавать вычисляемые поля прямо в запросе.

In [ ]:
run_sql('''
SELECT
    product_name,
    price,
    quantity,
    price * quantity AS total_amount
FROM orders;
''')

,product_name,price,quantity,total_amount
0,Product A,10.0,2,20.0
1,Product B,20.0,1,20.0
2,Product C,15.0,3,45.0
3,Product D,12.0,2,24.0
4,Product E,25.0,1,25.0


`AS total_amount` — это псевдоним для вычисляемого столбца.

Теперь отсортируем заказы по сумме заказа по убыванию.

In [ ]:
run_sql('''
SELECT
    product_name,
    price,
    quantity,
    price * quantity AS total_amount
FROM orders
ORDER BY total_amount DESC;
''')

,product_name,price,quantity,total_amount
0,Product C,15.0,3,45.0
1,Product E,25.0,1,25.0
2,Product D,12.0,2,24.0
3,Product A,10.0,2,20.0
4,Product B,20.0,1,20.0


## Обсуждение

Ответьте на вопросы:

1. Почему сумма заказа отличается от цены товара?
2. Что делает выражение `price * quantity`?
3. Для чего нужен псевдоним `AS total_amount`?
4. Как найти самый дорогой заказ?

# 9. INNER JOIN

`INNER JOIN` объединяет таблицы и показывает только те строки, для которых есть совпадение в обеих таблицах.

В нашем случае:

- из таблицы `users` берутся пользователи;
- из таблицы `orders` берутся их заказы;
- соединение идёт по условию `users.id = orders.user_id`.

Если пользователь не сделал заказ, он не попадёт в результат `INNER JOIN`.

In [ ]:
run_sql('''
SELECT
    users.name,
    users.email,
    orders.product_name,
    orders.order_date
FROM users
INNER JOIN orders
    ON users.id = orders.user_id;
''')

,name,email,product_name,order_date
0,Ivan,ivan@example.com,Product A,2023-02-22
1,Gleb,gleb@example.com,Product B,2023-02-20
2,Gleb,gleb@example.com,Product C,2023-02-23
3,Ivan,ivan@example.com,Product D,2023-02-25
4,Gleb,gleb@example.com,Product E,2023-02-26


## Задание 1. Добавить сумму заказа в INNER JOIN

Теперь добавим вычисляемое поле `total_amount`.

In [ ]:
run_sql('''
SELECT
    users.name,
    users.email,
    orders.product_name,
    orders.price,
    orders.quantity,
    orders.price * orders.quantity AS total_amount,
    orders.order_date
FROM users
INNER JOIN orders
    ON users.id = orders.user_id;
''')

,name,email,product_name,price,quantity,total_amount,order_date
0,Ivan,ivan@example.com,Product A,10.0,2,20.0,2023-02-22
1,Gleb,gleb@example.com,Product B,20.0,1,20.0,2023-02-20
2,Gleb,gleb@example.com,Product C,15.0,3,45.0,2023-02-23
3,Ivan,ivan@example.com,Product D,12.0,2,24.0,2023-02-25
4,Gleb,gleb@example.com,Product E,25.0,1,25.0,2023-02-26


## Обсуждение

Ответьте на вопросы:

1. Что делает `INNER JOIN`?
2. Почему в результате нет пользователей без заказов?
3. Что означает условие `ON users.id = orders.user_id`?
4. Почему при работе с двумя таблицами удобно писать `users.name`, `orders.product_name`?

# 10. LEFT JOIN

`LEFT JOIN` показывает:

1. все строки из левой таблицы;
2. совпадающие строки из правой таблицы;
3. если совпадений нет, в полях правой таблицы будет `NULL`.

В нашем запросе левая таблица — `users`, правая таблица — `orders`.

Значит, `LEFT JOIN` покажет всех пользователей, даже если у них нет заказов.

In [ ]:
run_sql('''
SELECT
    users.name,
    users.email,
    orders.product_name,
    orders.order_date
FROM users
LEFT JOIN orders
    ON users.id = orders.user_id;
''')

,name,email,product_name,order_date
0,Ivan,ivan@example.com,Product A,2023-02-22
1,Ivan,ivan@example.com,Product D,2023-02-25
2,Gleb,gleb@example.com,Product B,2023-02-20
3,Gleb,gleb@example.com,Product C,2023-02-23
4,Gleb,gleb@example.com,Product E,2023-02-26
5,Sergey,sergey@example.com,None,None
6,Andrew,andrew@example.com,None,None
7,Bob,bob@example.com,None,None
8,Tom,tom@example.com,None,None


Обратите внимание: у пользователей без заказов в полях `product_name` и `order_date` стоят значения `None`.  
В SQL это соответствует `NULL`.

## Сравнение INNER JOIN и LEFT JOIN

| Тип JOIN | Что показывает |
|---|---|
| INNER JOIN | только пользователей, у которых есть заказы |
| LEFT JOIN | всех пользователей, даже если заказов нет |

Это важное различие для аналитики.

Например:

- `INNER JOIN` удобен, когда анализируем только активных покупателей;
- `LEFT JOIN` удобен, когда нужно увидеть всех пользователей и найти неактивных.

# 11. Поиск пользователей без заказов

Одна из частых аналитических задач — найти объекты, у которых нет связанных действий.

Примеры:

- клиенты без заказов;
- студенты без посещений;
- товары без продаж;
- сотрудники без задач.

Для этого используется `LEFT JOIN` и проверка `IS NULL`.

In [ ]:
run_sql('''
SELECT
    users.id,
    users.name,
    users.email
FROM users
LEFT JOIN orders
    ON users.id = orders.user_id
WHERE orders.id IS NULL;
''')

,id,name,email
0,3,Sergey,sergey@example.com
1,4,Andrew,andrew@example.com
2,5,Bob,bob@example.com
3,6,Tom,tom@example.com


Почему это работает:

1. `LEFT JOIN` сначала показывает всех пользователей;
2. если у пользователя нет заказа, поля из `orders` становятся `NULL`;
3. условие `WHERE orders.id IS NULL` оставляет только пользователей без заказов.

## Обсуждение

Ответьте на вопросы:

1. Почему для поиска пользователей без заказов нужен `LEFT JOIN`, а не `INNER JOIN`?
2. Что означает `IS NULL`?
3. Где в бизнес-аналитике может пригодиться такой запрос?

# 12. Общая сумма заказов по каждому пользователю

Теперь решим более аналитическую задачу.

Нужно посчитать, на какую сумму каждый пользователь сделал заказы.

Для этого нужны:

- `INNER JOIN`;
- вычисляемое поле `price * quantity`;
- агрегатная функция `SUM`;
- группировка `GROUP BY`.

In [ ]:
run_sql('''
SELECT
    users.name,
    SUM(orders.price * orders.quantity) AS total_spent
FROM users
INNER JOIN orders
    ON users.id = orders.user_id
GROUP BY users.name
ORDER BY total_spent DESC;
''')

,name,total_spent
0,Gleb,90.0
1,Ivan,44.0


Этот запрос уже похож на реальную задачу аналитика: мы соединяем таблицы, считаем показатель и сортируем результат.

## Задание 2. Посчитать количество заказов по каждому пользователю

In [ ]:
run_sql('''
SELECT
    users.name,
    COUNT(orders.id) AS orders_count
FROM users
INNER JOIN orders
    ON users.id = orders.user_id
GROUP BY users.name
ORDER BY orders_count DESC;
''')

,name,orders_count
0,Gleb,3
1,Ivan,2


## Задание 3. Посчитать среднюю сумму заказа по каждому пользователю

In [ ]:
run_sql('''
SELECT
    users.name,
    AVG(orders.price * orders.quantity) AS avg_order_amount
FROM users
INNER JOIN orders
    ON users.id = orders.user_id
GROUP BY users.name
ORDER BY avg_order_amount DESC;
''')

,name,avg_order_amount
0,Gleb,30.0
1,Ivan,22.0


# 13. Агрегация с LEFT JOIN

Если использовать `INNER JOIN`, в результат попадут только пользователи с заказами.

Если нужно показать всех пользователей, включая тех, у кого нет заказов, используем `LEFT JOIN`.

Посчитаем количество заказов по каждому пользователю, включая пользователей без заказов.

In [ ]:
run_sql('''
SELECT
    users.name,
    COUNT(orders.id) AS orders_count
FROM users
LEFT JOIN orders
    ON users.id = orders.user_id
GROUP BY users.name
ORDER BY orders_count DESC;
''')

,name,orders_count
0,Gleb,3
1,Ivan,2
2,Tom,0
3,Sergey,0
4,Bob,0
5,Andrew,0


Обратите внимание: у пользователей без заказов `orders_count` равен 0.

Это удобно для анализа активности пользователей.

# 14. Итоговый мини-кейс

## Кейс: «Анализ пользователей и заказов»

Представим, что мы аналитики интернет-сервиса.

У нас есть две таблицы:

1. `users` — пользователи сервиса;
2. `orders` — заказы пользователей.

Нужно выполнить базовый анализ и подготовить выводы.

## Задания мини-кейса

Выполните SQL-запросы:

1. Вывести всех пользователей.
2. Вывести все заказы.
3. Вывести заказы с рассчитанной суммой заказа.
4. Объединить пользователей и заказы через `INNER JOIN`.
5. Объединить пользователей и заказы через `LEFT JOIN`.
6. Найти пользователей без заказов.
7. Посчитать общую сумму заказов по каждому пользователю.
8. Посчитать количество заказов по каждому пользователю.
9. Посчитать среднюю сумму заказа по каждому пользователю.
10. Сформулировать 2–3 аналитических вывода.

## Решение мини-кейса

### 1. Все пользователи

In [ ]:
run_sql('''
SELECT *
FROM users;
''')

,id,name,email,age,country,balance
0,1,Ivan,ivan@example.com,25,RUS,100000.0
1,2,Gleb,gleb@example.com,26,RUS,90000.0
2,3,Sergey,sergey@example.com,28,RUS,90000.0
3,4,Andrew,andrew@example.com,24,RUS,95000.0
4,5,Bob,bob@example.com,25,USA,100000.0
5,6,Tom,tom@example.com,29,USA,110000.0


### 2. Все заказы

In [ ]:
run_sql('''
SELECT *
FROM orders;
''')

,id,user_id,product_name,price,quantity,order_date
0,1,1,Product A,10.0,2,2023-02-22
1,2,2,Product B,20.0,1,2023-02-20
2,3,2,Product C,15.0,3,2023-02-23
3,4,1,Product D,12.0,2,2023-02-25
4,5,2,Product E,25.0,1,2023-02-26


### 3. Заказы с рассчитанной суммой заказа

In [ ]:
run_sql('''
SELECT
    id,
    user_id,
    product_name,
    price,
    quantity,
    price * quantity AS total_amount,
    order_date
FROM orders;
''')

,id,user_id,product_name,price,quantity,total_amount,order_date
0,1,1,Product A,10.0,2,20.0,2023-02-22
1,2,2,Product B,20.0,1,20.0,2023-02-20
2,3,2,Product C,15.0,3,45.0,2023-02-23
3,4,1,Product D,12.0,2,24.0,2023-02-25
4,5,2,Product E,25.0,1,25.0,2023-02-26


### 4. Пользователи и заказы через INNER JOIN

In [ ]:
run_sql('''
SELECT
    users.name,
    users.email,
    orders.product_name,
    orders.price * orders.quantity AS total_amount,
    orders.order_date
FROM users
INNER JOIN orders
    ON users.id = orders.user_id;
''')

,name,email,product_name,total_amount,order_date
0,Ivan,ivan@example.com,Product A,20.0,2023-02-22
1,Gleb,gleb@example.com,Product B,20.0,2023-02-20
2,Gleb,gleb@example.com,Product C,45.0,2023-02-23
3,Ivan,ivan@example.com,Product D,24.0,2023-02-25
4,Gleb,gleb@example.com,Product E,25.0,2023-02-26


### 5. Пользователи и заказы через LEFT JOIN

In [ ]:
run_sql('''
SELECT
    users.name,
    users.email,
    orders.product_name,
    orders.order_date
FROM users
LEFT JOIN orders
    ON users.id = orders.user_id;
''')

,name,email,product_name,order_date
0,Ivan,ivan@example.com,Product A,2023-02-22
1,Ivan,ivan@example.com,Product D,2023-02-25
2,Gleb,gleb@example.com,Product B,2023-02-20
3,Gleb,gleb@example.com,Product C,2023-02-23
4,Gleb,gleb@example.com,Product E,2023-02-26
5,Sergey,sergey@example.com,None,None
6,Andrew,andrew@example.com,None,None
7,Bob,bob@example.com,None,None
8,Tom,tom@example.com,None,None


### 6. Пользователи без заказов

In [ ]:
run_sql('''
SELECT
    users.name,
    users.email
FROM users
LEFT JOIN orders
    ON users.id = orders.user_id
WHERE orders.id IS NULL;
''')

,name,email
0,Sergey,sergey@example.com
1,Andrew,andrew@example.com
2,Bob,bob@example.com
3,Tom,tom@example.com


### 7. Общая сумма заказов по каждому пользователю

In [ ]:
run_sql('''
SELECT
    users.name,
    SUM(orders.price * orders.quantity) AS total_spent
FROM users
INNER JOIN orders
    ON users.id = orders.user_id
GROUP BY users.name
ORDER BY total_spent DESC;
''')

,name,total_spent
0,Gleb,90.0
1,Ivan,44.0


### 8. Количество заказов по каждому пользователю

In [ ]:
run_sql('''
SELECT
    users.name,
    COUNT(orders.id) AS orders_count
FROM users
LEFT JOIN orders
    ON users.id = orders.user_id
GROUP BY users.name
ORDER BY orders_count DESC;
''')

,name,orders_count
0,Gleb,3
1,Ivan,2
2,Tom,0
3,Sergey,0
4,Bob,0
5,Andrew,0


### 9. Средняя сумма заказа по каждому пользователю

In [ ]:
run_sql('''
SELECT
    users.name,
    AVG(orders.price * orders.quantity) AS avg_order_amount
FROM users
INNER JOIN orders
    ON users.id = orders.user_id
GROUP BY users.name
ORDER BY avg_order_amount DESC;
''')

,name,avg_order_amount
0,Gleb,30.0
1,Ivan,22.0


## Пример аналитических выводов

По результатам мини-кейса можно сделать следующие выводы:

1. Заказы есть только у части пользователей: у Ivan и Gleb, остальные пользователи не совершали заказов.
2. Пользователь Gleb сделал больше всего заказов и имеет наибольшую общую сумму покупок.
3. `LEFT JOIN` позволяет выявить пользователей без заказов, что может быть полезно для анализа неактивной аудитории.
4. Связка `JOIN + GROUP BY + SUM` позволяет получить полноценный аналитический отчёт по активности пользователей.

# 15. Дополнительное задание повышенной сложности

Это задание можно выполнить, если осталось время.

## Задача

Нужно получить таблицу со всеми пользователями и следующими показателями:

- имя пользователя;
- страна;
- количество заказов;
- общая сумма заказов;
- средняя сумма заказа.

Пользователи без заказов тоже должны попасть в результат.

In [ ]:
run_sql('''
SELECT
    users.name,
    users.country,
    COUNT(orders.id) AS orders_count,
    COALESCE(SUM(orders.price * orders.quantity), 0) AS total_spent,
    COALESCE(AVG(orders.price * orders.quantity), 0) AS avg_order_amount
FROM users
LEFT JOIN orders
    ON users.id = orders.user_id
GROUP BY users.name, users.country
ORDER BY total_spent DESC;
''')

,name,country,orders_count,total_spent,avg_order_amount
0,Gleb,RUS,3,90.0,30.0
1,Ivan,RUS,2,44.0,22.0
2,Andrew,RUS,0,0.0,0.0
3,Bob,USA,0,0.0,0.0
4,Sergey,RUS,0,0.0,0.0
5,Tom,USA,0,0.0,0.0


## Что здесь нового

`COALESCE()` заменяет `NULL` на указанное значение.

Например:

```sql
COALESCE(SUM(...), 0)
```

Если у пользователя нет заказов, сумма будет не `NULL`, а `0`.

Это удобно для отчётов.

# 16. Частые ошибки при работе с JOIN

## Ошибка 1. Забыли условие ON

Неправильно:

```sql
SELECT *
FROM users
INNER JOIN orders;
```

Правильно:

```sql
SELECT *
FROM users
INNER JOIN orders
    ON users.id = orders.user_id;
```

---

## Ошибка 2. Перепутали INNER JOIN и LEFT JOIN

Если нужно найти пользователей без заказов, `INNER JOIN` не подойдёт, потому что он покажет только совпадающие записи.

Для поиска отсутствующих связанных данных нужен `LEFT JOIN`.

---

## Ошибка 3. Не указали имя таблицы перед столбцом

Если в двух таблицах есть одинаковые названия столбцов, например `id`, лучше писать явно:

```sql
users.id
orders.id
```

Так запрос становится понятнее и безопаснее.

---

## Ошибка 4. Забыли GROUP BY при агрегации

Если нужно посчитать сумму заказов по каждому пользователю, нужен `GROUP BY`.

Правильно:

```sql
SELECT users.name, SUM(orders.price * orders.quantity)
FROM users
INNER JOIN orders
    ON users.id = orders.user_id
GROUP BY users.name;
```

# 17. Контрольные вопросы по четвёртой паре

Ответьте на вопросы:

1. Зачем данные разделяют на несколько таблиц?
2. Что такое первичный ключ?
3. Что такое внешний ключ?
4. Как связаны таблицы `users` и `orders`?
5. Что делает `INNER JOIN`?
6. Что делает `LEFT JOIN`?
7. Чем `INNER JOIN` отличается от `LEFT JOIN`?
8. Что означает `NULL`?
9. Как найти пользователей без заказов?
10. Как рассчитать сумму заказа?
11. Как посчитать общую сумму заказов по каждому пользователю?
12. Зачем использовать `GROUP BY` вместе с `JOIN`?
13. Для чего нужна функция `COALESCE()`?
14. Какие аналитические выводы можно сделать по таблицам `users` и `orders`?

# 18. Итог всей темы

За четыре пары мы прошли полный путь от сбора данных до базового SQL-анализа.

## Пара 1

Мы изучили:

- CSV;
- JSON;
- DataFrame;
- первичную проверку данных;
- простые расчёты и группировки.

## Пара 2

Мы изучили:

- HTML;
- web scraping;
- BeautifulSoup;
- извлечение данных из HTML;
- очистку и сохранение результатов.

## Пара 3

Мы изучили:

- основы SQL;
- создание таблицы;
- `SELECT`;
- `WHERE`;
- `ORDER BY`;
- агрегатные функции;
- `GROUP BY`.

## Пара 4

Мы изучили:

- работу с несколькими таблицами;
- связи между таблицами;
- `INNER JOIN`;
- `LEFT JOIN`;
- поиск записей без связанных данных;
- итоговый аналитический мини-кейс.

---

## Главная логика модуля

```text
Источник данных → Таблица → База данных → SQL-запрос → Аналитический вывод
```

Именно эта логика лежит в основе работы аналитика данных.

# 19. Финальное задание для самостоятельного закрепления

Сформулируйте короткий аналитический отчёт по результатам работы с таблицами `users` и `orders`.

В отчёте нужно указать:

1. сколько всего пользователей;
2. сколько всего заказов;
3. какие пользователи делали заказы;
4. какие пользователи не делали заказов;
5. кто сделал заказов больше всего;
6. кто потратил больше всего;
7. какой вывод можно сделать по активности пользователей.

Пример структуры ответа:

```text
В таблице users содержится 6 пользователей. В таблице orders содержится 5 заказов.
Заказы совершали пользователи Ivan и Gleb. Остальные пользователи заказов не имеют.
Наибольшее количество заказов сделал Gleb. Он же имеет наибольшую общую сумму покупок.
Таким образом, часть пользователей активна, а часть не совершает заказов, что может быть основой для дальнейшего анализа клиентской активности.
```